In [27]:
import pandas as pd
import time
from datetime import datetime
from pandas.api.types import is_datetime64_any_dtype

StatementMeta(, 751c80a2-1394-4839-8b2b-2c6333838ce7, 29, Finished, Available, Finished)

In [28]:
reviews = pd.read_csv("/lakehouse/default/Files/olist_order_reviews_dataset.csv")
print(reviews.head())

StatementMeta(, 751c80a2-1394-4839-8b2b-2c6333838ce7, 30, Finished, Available, Finished)

                          review_id                          order_id  \
0  7bc2406110b926393aa56f80a40eba40  73fc7af87114b39712e6da79b0a377eb   
1  80e641a11e56f04c1ad469d5645fdfde  a548910a1c6147796b98fdf73dbeba33   
2  228ce5500dc1d8e020d8d1322874b6f0  f9e4b658b201a9f2ecdecbb34bed034b   
3  e64fb393e7b32834bb789ff8bb30750e  658677c97b385a9be170737859d3511b   
4  f7c4243c7fe1938f181bec41a392bdeb  8e6bfb81e283fa7e4f11123a3fb894f1   

   review_score review_comment_title  \
0             4                  NaN   
1             5                  NaN   
2             5                  NaN   
3             5                  NaN   
4             5                  NaN   

                              review_comment_message review_creation_date  \
0                                                NaN  2018-01-18 00:00:00   
1                                                NaN  2018-03-10 00:00:00   
2                                                NaN  2018-02-17 00:00:00   
3           

In [29]:
print(reviews.info())

StatementMeta(, 751c80a2-1394-4839-8b2b-2c6333838ce7, 31, Finished, Available, Finished)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 99224 entries, 0 to 99223
Data columns (total 7 columns):
 #   Column                   Non-Null Count  Dtype 
---  ------                   --------------  ----- 
 0   review_id                99224 non-null  object
 1   order_id                 99224 non-null  object
 2   review_score             99224 non-null  int64 
 3   review_comment_title     11568 non-null  object
 4   review_comment_message   40977 non-null  object
 5   review_creation_date     99224 non-null  object
 6   review_answer_timestamp  99224 non-null  object
dtypes: int64(1), object(6)
memory usage: 5.3+ MB
None


In [30]:
# Convert review_creation_date and review_answer_timestamp to datetime objects

reviews['review_creation_date'] = pd.to_datetime(reviews['review_creation_date'])
reviews['review_answer_timestamp'] = pd.to_datetime(reviews['review_answer_timestamp'])

StatementMeta(, 751c80a2-1394-4839-8b2b-2c6333838ce7, 32, Finished, Available, Finished)

In [31]:
# verify the datatype has been changed
assert is_datetime64_any_dtype(reviews['review_creation_date']), "Column 'review_creation_date' is not of datetime type"
assert is_datetime64_any_dtype(reviews['review_answer_timestamp']), "Column 'review_answer_timestamp' is not of datetime type"

StatementMeta(, 751c80a2-1394-4839-8b2b-2c6333838ce7, 33, Finished, Available, Finished)

In [32]:
print(reviews.info())

StatementMeta(, 751c80a2-1394-4839-8b2b-2c6333838ce7, 34, Finished, Available, Finished)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 99224 entries, 0 to 99223
Data columns (total 7 columns):
 #   Column                   Non-Null Count  Dtype         
---  ------                   --------------  -----         
 0   review_id                99224 non-null  object        
 1   order_id                 99224 non-null  object        
 2   review_score             99224 non-null  int64         
 3   review_comment_title     11568 non-null  object        
 4   review_comment_message   40977 non-null  object        
 5   review_creation_date     99224 non-null  datetime64[ns]
 6   review_answer_timestamp  99224 non-null  datetime64[ns]
dtypes: datetime64[ns](2), int64(1), object(4)
memory usage: 5.3+ MB
None


In [33]:
# Check for nulls

null_counts = reviews.isnull().sum()
print(null_counts)

StatementMeta(, 751c80a2-1394-4839-8b2b-2c6333838ce7, 35, Finished, Available, Finished)

review_id                      0
order_id                       0
review_score                   0
review_comment_title       87656
review_comment_message     58247
review_creation_date           0
review_answer_timestamp        0
dtype: int64


In [43]:
# Check for duplicate review_id
duplicate_reviews = reviews['review_id'].duplicated()
print("Boolean Series for duplicates in 'review_id':")
print(duplicate_reviews)

# Display the actual duplicate values in 'review_id'
duplicate_review_id = reviews[reviews['review_id'].duplicated(keep=False)]['review_id']   # This excludes the first occurence
print("\nDuplicate values in 'review_id':")
print(duplicate_review_id)

StatementMeta(, 751c80a2-1394-4839-8b2b-2c6333838ce7, 45, Finished, Available, Finished)

Boolean Series for duplicates in 'review_id':
0        False
1        False
2        False
3        False
4        False
         ...  
99219    False
99220    False
99221    False
99222    False
99223    False
Name: review_id, Length: 99224, dtype: bool

Duplicate values in 'review_id':
200      28642ce6250b94cc72bc85960aec6c62
344      a0a641414ff718ca079b3967ef5c2495
346      f4d74b17cd63ee35efa82cd2567de911
360      ecbaf1fce7d2c09bfab46f89065afeaf
393      6b1de94de0f4bd84dfc4136818242faa
                       ...               
99108    2c6c08892b83ba4c1be33037c2842294
99124    6ec93e77f444e0b1703740a69122e35d
99164    2afe63a67dfd99b3038f568fb47ee761
99167    017808d29fd1f942d97e50184dfb4c13
99178    44d1e9165ec54b1d89d33594856af859
Name: review_id, Length: 1603, dtype: object


##### Since there are 1603 duplicated review_ids, let's investigate further by checking their order status and if the number of order items contributes to the duplication in the review_id. 

In [37]:
# Load Silver tables
orders_silver= spark.read.table("SilverLakehouse.dbo.olist_orders_cleaned")
items_silver = spark.read.table("SilverLakehouse.dbo.olist_items_cleaned")

StatementMeta(, 751c80a2-1394-4839-8b2b-2c6333838ce7, 39, Finished, Available, Finished)

In [38]:
# Convert the tables to pandas dataframe

orders = orders_silver.toPandas()
items = items_silver.toPandas()

StatementMeta(, 751c80a2-1394-4839-8b2b-2c6333838ce7, 40, Finished, Available, Finished)

In [42]:
# Extract only the duplicate review_id rows

# Find duplicated review_ids (marking all occurrences)
dup_mask = reviews['review_id'].duplicated(keep=False)

# Extract all rows (all columns) for those duplicated IDs
duplicated_reviews = reviews[dup_mask]

# Optional: sort by review_id for easier inspection
duplicated_reviews = duplicated_reviews.sort_values('review_id')

print(duplicated_reviews.head())
print(f"Total duplicated rows: {len(duplicated_reviews)}")


StatementMeta(, 751c80a2-1394-4839-8b2b-2c6333838ce7, 44, Finished, Available, Finished)

                              review_id                          order_id  \
46678  00130cbe1f9d422698c812ed8ded1919  dfcdfc43867d1c1381bfaf62d6b9c195   
29841  00130cbe1f9d422698c812ed8ded1919  04a28263e085d399c97ae49e0b477efa   
90677  0115633a9c298b6a98bcbe4eee75345f  78a4201f58af3463bdab842eea4bc801   
63193  0115633a9c298b6a98bcbe4eee75345f  0c9850b2c179c1ef60d2855e2751d1fa   
92876  0174caf0ee5964646040cd94e15ac95e  f93a732712407c02dce5dd5088d0f47b   

       review_score review_comment_title  \
46678             1                  NaN   
29841             1                  NaN   
90677             5                  NaN   
63193             5                  NaN   
92876             1                  NaN   

                                  review_comment_message review_creation_date  \
46678  O cartucho "original HP" 60XL não é reconhecid...           2018-03-07   
29841  O cartucho "original HP" 60XL não é reconhecid...           2018-03-07   
90677                        

In [44]:
# Join with items to see if order_item_id was more than 1 for these duplicated review_ids

# Join duplicated reviews with items on order_id
reviews_items_joined = duplicated_reviews.merge(
    items,
    on="order_id",
    how="left",
    suffixes=("_review", "_item")
)

# Count how many items each order has
item_counts = items.groupby("order_id").size().reset_index(name="item_count")

# Merge this count back to see the number of items per order
reviews_with_itemcount = duplicated_reviews.merge(
    item_counts,
    on="order_id",
    how="left"
)

# Optional: check where there’s more than 1 item per order
multiple_item_orders = reviews_with_itemcount[reviews_with_itemcount["item_count"] > 1]

print(f"Total duplicated reviews with multiple items: {len(multiple_item_orders)}")
print(multiple_item_orders.head())

StatementMeta(, 751c80a2-1394-4839-8b2b-2c6333838ce7, 46, Finished, Available, Finished)

Total duplicated reviews with multiple items: 198
                           review_id                          order_id  \
3   0115633a9c298b6a98bcbe4eee75345f  0c9850b2c179c1ef60d2855e2751d1fa   
19  0467560f511c516ddaa54a60edb0c291  55b2e390d5d80ada31ad1b795ebeb087   
24  0501aab2f381486c36bf0f071442c0c2  0068c109948b9a1dfb8530d1978acef3   
30  06e327fb381850fdd69fba40ad61b2f2  46d2741b78a72c2f07bf0f3c34c4ceab   
43  09534ea272eea99209ff37c5c348ceb1  4cf18bf9d25331902ba212fa69a7a01d   

    review_score review_comment_title  \
3              5                  NaN   
19             5                  NaN   
24             1                  NaN   
30             2                  NaN   
43             5                  NaN   

                               review_comment_message review_creation_date  \
3                                                 NaN           2017-09-21   
19                                                NaN           2017-02-09   
24  Espero obter uma res

In [55]:
# Merge with orders to find the order status of these 98 reviews

# Get item count per order
item_counts = items.groupby("order_id").size().reset_index(name="item_count")

# Merge duplicate reviews with item counts
reviews_with_itemcount = duplicated_reviews.merge(
    item_counts,
    on="order_id",
    how="left"
)

# Filter only reviews with multiple items
multiple_item_reviews = reviews_with_itemcount[reviews_with_itemcount["item_count"] > 1]

# Merge with orders to get order status
multiple_item_reviews_with_status = multiple_item_reviews.merge(
    orders[['order_id', 'order_status']],
    on='order_id',
    how='left'
)

# Check results
print(multiple_item_reviews_with_status[['review_id', 'order_id', 'order_status', 'item_count']].head())
print(multiple_item_reviews_with_status['order_status'].value_counts())
print(multiple_item_reviews_with_status['item_count'].value_counts())


StatementMeta(, 751c80a2-1394-4839-8b2b-2c6333838ce7, 57, Finished, Available, Finished)

                          review_id                          order_id  \
0  0115633a9c298b6a98bcbe4eee75345f  0c9850b2c179c1ef60d2855e2751d1fa   
1  0467560f511c516ddaa54a60edb0c291  55b2e390d5d80ada31ad1b795ebeb087   
2  0501aab2f381486c36bf0f071442c0c2  0068c109948b9a1dfb8530d1978acef3   
3  06e327fb381850fdd69fba40ad61b2f2  46d2741b78a72c2f07bf0f3c34c4ceab   
4  09534ea272eea99209ff37c5c348ceb1  4cf18bf9d25331902ba212fa69a7a01d   

  order_status  item_count  
0    delivered         2.0  
1    delivered         2.0  
2    delivered         2.0  
3    delivered         3.0  
4    delivered         2.0  
order_status
delivered    194
canceled       2
shipped        1
invoiced       1
Name: count, dtype: int64
item_count
2.0     149
3.0      33
4.0       9
6.0       3
5.0       3
11.0      1
Name: count, dtype: int64


##### Of the 1603 duplicates, 198 reviews are due to multiple items within the same order_id. Of these 198, we may consider 196 as valid orders. 2 are canceled.
##### We have another 1409 left to investigate.

In [46]:
# Join duplicated_reviews with orders to check on their order status. 

# Get order_ids of multi-item orders
multi_item_order_ids = multiple_item_reviews_with_status['order_id'].unique()

# Filter out reviews belonging to multi-item orders
remaining_dupe_reviews = duplicated_reviews[~duplicated_reviews['order_id'].isin(multi_item_order_ids)]

# Join with orders to check their order status
remaining_dupe_with_status = remaining_dupe_reviews.merge(
    orders[['order_id', 'order_status']],
    on='order_id',
    how='left'
)

# Get a summary of order statuses
status_summary = (
    remaining_dupe_with_status['order_status']
    .value_counts(dropna=False)
    .reset_index()
    .rename(columns={'index': 'order_status', 'order_status': 'count'})
)

print(status_summary)

StatementMeta(, 751c80a2-1394-4839-8b2b-2c6333838ce7, 48, Finished, Available, Finished)

         count  count
0    delivered   1298
1     canceled     59
2      shipped     27
3  unavailable     10
4   processing      6
5     invoiced      5


In [49]:
# Step 1 — Filter out the 198 multi-item orders from duplicates
multi_item_order_ids = multiple_item_reviews_with_status['order_id'].unique()
remaining_dupes = duplicated_reviews[~duplicated_reviews['order_id'].isin(multi_item_order_ids)]

# Step 2 — Join with orders to get order_status and customer_id
remaining_dupes_with_status = remaining_dupes.merge(
    orders[['order_id', 'order_status', 'customer_id']],
    on='order_id',
    how='left'
)

# Step 3 — Select columns we want to inspect
inspect_dupes = remaining_dupes_with_status[[
    'review_id',
    'order_id',
    'customer_id',
    'review_creation_date',
    'review_comment_message',
    'order_status'
]]

# Step 4 — Sort by review_id so duplicates are together
inspect_dupes = inspect_dupes.sort_values(by=['review_id', 'review_creation_date'])

# Step 5 — View the result
display(inspect_dupes)

StatementMeta(, 751c80a2-1394-4839-8b2b-2c6333838ce7, 51, Finished, Available, Finished)

SynapseWidget(Synapse.DataFrame, 4b1f2806-83d2-4368-b6a3-5d73db4d55a5)

In [50]:
# Check for review_ids with same customer id. 

# Group by review_id and count unique customer_ids
customer_check = inspect_dupes.groupby('review_id')['customer_id'].nunique().reset_index()

# Flag review_ids where customer_id differs
customer_check['different_customers'] = customer_check['customer_id'] > 1

# See only review_ids where customer_id is not consistent
conflicting_reviews = customer_check[customer_check['different_customers']]

print(f"Number of review_ids with different customer_ids: {len(conflicting_reviews)}")
display(conflicting_reviews)

StatementMeta(, 751c80a2-1394-4839-8b2b-2c6333838ce7, 52, Finished, Available, Finished)

Number of review_ids with different customer_ids: 617


SynapseWidget(Synapse.DataFrame, 2f8ec364-10d0-4f64-b001-eb458ac8f61d)

In [62]:
# Of the 788 with the same customer_id and review_id, let's check if they have same review_creation_date and message

# Step 1 — Find review_ids where customer_id is the same
same_customer_reviews = customer_check[customer_check['different_customers'] == False]['review_id']

# Step 2 — Filter inspect_dupes for these review_ids
same_customer_df = inspect_dupes[inspect_dupes['review_id'].isin(same_customer_reviews)]

# Step 3 — Group by review_id to check uniqueness of message, creation date, order_id, and order_status
message_date_check = same_customer_df.groupby('review_id').agg(
    unique_messages      = ('review_comment_message', lambda x: x.nunique()),
    unique_dates         = ('review_creation_date', lambda x: x.nunique()),
    unique_order_ids     = ('order_id', lambda x: x.nunique()),
    unique_order_status  = ('order_status', lambda x: x.nunique())
).reset_index()

# Step 4 — Flag review_ids where all these are identical
message_date_check['same_message_date_order_status'] = (
    (message_date_check['unique_messages'] == 1) &
    (message_date_check['unique_dates'] == 1) &
    (message_date_check['unique_order_ids'] == 1) &
    (message_date_check['unique_order_status'] == 1)
)

# Step 5 — See summary
print(f"Total review_id groups with same customer_id: {len(same_customer_reviews)}")  # Distinct review_ids
print(f"Of these, identical message, date, order_id, order_status: {message_date_check['same_message_date_order_status'].sum()}")
print(f"Non-identical groups count: {len(message_date_check) - message_date_check['same_message_date_order_status'].sum()}")

display(message_date_check)


StatementMeta(, 751c80a2-1394-4839-8b2b-2c6333838ce7, 64, Finished, Available, Finished)

Total review_id groups with same customer_id: 151
Of these, identical message, date, order_id, order_status: 65
Non-identical groups count: 86


SynapseWidget(Synapse.DataFrame, d1cab2d2-76d2-4f0e-b5fb-5fb8b84e18cf)

##### Of the 151 distinct review_ids (with only one item per order), 65 of them have same review_id, customer_id, review_creation_date and review_message. These can be considered as true duplicates and we can remove them. 

In [63]:
# Step 1 — Identify the 65 review_ids
ids_to_remove = message_date_check.loc[message_date_check['same_message_and_date'], 'review_id']

# Step 2 — Count how many rows each of these review_ids has
rows_to_remove = inspect_dupes[inspect_dupes['review_id'].isin(ids_to_remove)]
print(f"Rows to remove: {rows_to_remove.shape[0]}")  # should be 65 or more

# Step 3 — Remove these rows completely from inspect_dupes
cleaned_reviews = inspect_dupes[~inspect_dupes['review_id'].isin(ids_to_remove)].copy()

# Step 4 — Check results
print(f"Removed {rows_to_remove.shape[0]} rows in total.")
print(f"Remaining rows: {cleaned_reviews.shape[0]}")

display(cleaned_reviews.head())

StatementMeta(, 751c80a2-1394-4839-8b2b-2c6333838ce7, 65, Finished, Available, Finished)

KeyError: 'same_message_and_date'

In [58]:
# Step 1 — Identify the 65 review_ids to deduplicate
ids_to_dedupe = message_date_check.loc[message_date_check['same_message_and_date'], 'review_id']

# Step 2 — Keep only the first occurrence for those IDs, but keep all other rows
# Separate the dataset
to_dedupe = reviews[reviews['review_id'].isin(ids_to_dedupe)]
to_keep = reviews[~reviews['review_id'].isin(ids_to_dedupe)]

# Deduplicate only the identified set
to_dedupe_cleaned = to_dedupe.drop_duplicates(subset=['review_id'], keep='first')

# Step 3 — Combine back with the rest of the dataset
cleaned_reviews_original = pd.concat([to_keep, to_dedupe_cleaned], ignore_index=True)

# Step 4 — Check results
print(f"Original reviews rows: {reviews.shape[0]}")
print(f"Removed rows: {reviews.shape[0] - cleaned_reviews_original.shape[0]}")
print(f"Remaining rows: {cleaned_reviews_original.shape[0]}")

display(cleaned_reviews_original.head())


StatementMeta(, 751c80a2-1394-4839-8b2b-2c6333838ce7, 60, Finished, Available, Finished)

Original reviews rows: 99224
Removed rows: 66
Remaining rows: 99158


SynapseWidget(Synapse.DataFrame, df052c9d-2955-4003-8652-4c5846cd2489)

In [61]:
# Write the table to the silver lakehouse as a delta table
# Convert pandas to Spark
spark_reviews = spark.createDataFrame(cleaned_reviews_original)

# Save as a Delta table in Silver Lakehouse
silver_path = "SilverLakehouse.dbo.olist_reviews_cleaned"
spark_reviews.write.format("delta").mode("overwrite").saveAsTable(silver_path)

StatementMeta(, 751c80a2-1394-4839-8b2b-2c6333838ce7, 63, Finished, Available, Finished)

In [59]:
# Step 1 — Check for duplicate review_ids in the cleaned dataframe
duplicate_check = cleaned_reviews_original.duplicated(subset=['review_id'], keep=False)

print(f"Number of duplicate review_ids left: {duplicate_check.sum()}")

# Step 2 — See which review_ids still have duplicates
remaining_duplicates = cleaned_reviews_original[duplicate_check]['review_id'].unique()

print(f"Review_ids still duplicated: {len(remaining_duplicates)}")
print(remaining_duplicates)

# Step 3 — Spot check these review_ids if any
if len(remaining_duplicates) > 0:
    display(cleaned_reviews_original[cleaned_reviews_original['review_id'].isin(remaining_duplicates)])
else:
    print("✅ No remaining duplicate review_ids found.")

StatementMeta(, 751c80a2-1394-4839-8b2b-2c6333838ce7, 61, Finished, Available, Finished)

Number of duplicate review_ids left: 1472
Review_ids still duplicated: 724
['28642ce6250b94cc72bc85960aec6c62' 'a0a641414ff718ca079b3967ef5c2495'
 'f4d74b17cd63ee35efa82cd2567de911' 'ecbaf1fce7d2c09bfab46f89065afeaf'
 '6b1de94de0f4bd84dfc4136818242faa' '957011305e7a4b6c8a266eeeb8e0316d'
 'b34f4a786bf00b7c8486141ba482783b' 'd433c252647c51309432ca0b763f969b'
 '5f35bbde2b32b8617aab15b7a9a0ae24' 'c5976a5a98e854fb23d7e03c6754ae60'
 '62c7722239b976d943ec0d430cfe890e' 'ca8e5e40f55fbffaa631de686b627dde'
 '6ec93e77f444e0b1703740a69122e35d' 'ee9bde038f146b5ece0d9a859cc2a795'
 '830636803620cdf8b6ffaf1b2f6e92b2' '43e5286eb9f62cabe4471ec1dff7142a'
 '1f5aa5d7c3ee7895e4d223797b5a5f2d' '3242cc306a9218d0377831e175d62fbf'
 '1258a603cc9e4897f7955ab218c83b8c' '771654d19b26b9108952856b225ad105'
 'b703764760e6e5d534a8940a12fca101' '87c13a40ddb9333c0818f8f6745af183'
 'b06f6882b1c0bc82ddf057b58b86fe50' 'dbcd48d3ff3e8bc81c542ba3e20049cf'
 '48f315adb0fcfc986321083d894884a6' '5d1f78d752c653e019bd07cc48106300'
 '

SynapseWidget(Synapse.DataFrame, d51d66b1-8da9-4ab0-9c19-9004016b5235)